# 天気図パターン分類 - すぐに使う

リポジトリに同梱されている学習済みモデル(`weights/model.pt`)を使って、
天気図がどの気圧配置パターンに近いかを判定します。学習は行わず、
Google Driveのマウントも不要です。

上から順にセルを実行してください。

## 1. セットアップ

In [ ]:
REPO_URL = "https://github.com/awg-yk/weather-pattern-classification.git"
BRANCH = "claude/weather-chart-classification-4b6in1"
REPO_DIR = "/content/weather-pattern-classification"
WEIGHTS_PATH = f"{REPO_DIR}/weights/model.pt"  # リポジトリに同梱されているモデル

import subprocess, os

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

%cd {REPO_DIR}
!pip install -q -r requirements.txt
!apt-get -qq install -y fonts-noto-cjk poppler-utils

assert os.path.exists(WEIGHTS_PATH), f"モデルの重みが見つかりません: {WEIGHTS_PATH}"
print("セットアップ完了。モデル:", WEIGHTS_PATH)

In [ ]:
import sys
sys.path.append(REPO_DIR)

import matplotlib.pyplot as plt
from src.labels import LABEL_JA
from scripts.gradcam import explain_top_predictions
from scripts.fetch_and_predict import fetch_chart
from google.colab import files

TOP_K = 3


def classify_and_show(image_path: str):
    """画像1枚を分類し、入力画像+上位TOP_K件のヒートマップを表示、残りはテキストで出す。"""
    display_image, top_overlays, ranked = explain_top_predictions(
        image_path=image_path,
        weights_path=WEIGHTS_PATH,
        top_k=TOP_K,
        apply_preprocess=True,
    )

    fig, axes = plt.subplots(1, TOP_K + 1, figsize=(5 * (TOP_K + 1), 5))
    axes[0].imshow(display_image)
    axes[0].set_title("入力画像(前処理後)")
    axes[0].axis("off")

    for ax, (label, prob, overlay) in zip(axes[1:], top_overlays):
        ax.imshow(overlay)
        ax.set_title(f"{LABEL_JA[label]}\n({prob * 100:.1f}%)")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    print("--- 全ラベルの確信度 ---")
    for label, prob in ranked:
        print(f"{LABEL_JA[label]}: {prob * 100:.1f}%")


print("準備完了")

## 2. 画像を用意して分類

セル右側のフォームでMODEを選んでください。

- `upload`: 実行するとファイル選択ダイアログが出るので、手元の画像をアップロードする
- `date`: DATE(カレンダーから選択)・HOURで指定した気象庁の保存用天気図(JSMAP)を自動でダウンロードする
  - `HOUR`: `0`(日本時間9時)または`12`(日本時間21時)。この2つのみ存在する
  - 直近1年分程度、かつ月末から約3ヶ月遅れで追加されるアーカイブのため、
    古すぎる/新しすぎる日付は404になることがある

In [ ]:
#@markdown MODEを選び、dateを使う場合はDATEをカレンダーから選択してください。
MODE = "date" #@param ["upload", "date"]
DATE = '2025-01-01' #@param {type:"date"}
HOUR = 0 #@param [0, 12] {type:"raw"}

if MODE == "upload":
    uploaded = files.upload()
    image_path = list(uploaded.keys())[0]
elif MODE == "date":
    image_path = str(fetch_chart(DATE, hour=HOUR))
    print("取得:", image_path)
else:
    raise ValueError('MODEは "upload" か "date" を指定してください')

classify_and_show(image_path)